# 作业4：序列模型、RNN、双向RNN、词嵌入、注意力机制
日期：2026年6月17日
依赖库：numpy, torch, string

## 2 序列模型
### 2.1 理论计算题：一阶马尔可夫+拉普拉斯平滑
输入序列：`ababc`，词汇表 $V=\{a,b,c\}$
原始转移计数（前驱→后继）：
- $a\to b$: 2次
- $b\to a$: 1次
- $b\to c$: 1次
其余所有转移计数为0。
拉普拉斯平滑：所有转移计数 $+1$，分母 = 当前前驱所有后继平滑计数之和。

1. 求 $p(a|b)$
前驱`b`平滑后各后继计数：$a:1+1=2,\ b:0+1=1,\ c:1+1=2$，分母总和 $2+1+2=5$
$$p(a|b) = \frac{2}{5} = 0.4$$

2. 求 $p(c|b)$
$$p(c|b) = \frac{2}{5} = 0.4$$

### 2.2 编程题：文本预处理函数 preprocess_text

In [1]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除标点
    text_lower = text.lower()
    punc = set(string.punctuation)
    clean_chars = [c for c in text_lower if c not in punc]
    clean_text = ''.join(clean_chars)
    # 2. 空格分词
    tokens = clean_text.split()
    # 3. 构建词频词汇表，ID从0开始
    cnt = Counter(tokens)
    sorted_words = sorted(cnt.keys(), key=lambda x: (-cnt[x], x))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    # 4. 滑动窗口生成n长度特征序列与标签
    feat_list = []
    label_list = []
    for i in range(len(tokens) - n):
        feat = tokens[i:i+n]
        label = tokens[i+n]
        feat_list.append(feat)
        label_list.append(label)
    return vocab, (feat_list, label_list)

# 测试示例
if __name__ == '__main__':
    test_text = "The time machine"
    vocab, (feats, labels) = preprocess_text(test_text, n=2)
    print("词汇表:", vocab)
    print("特征序列:", feats)
    print("对应标签:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征序列: [['the', 'time']]
对应标签: ['machine']


## 3 循环神经网络
### 3.1 理论计算题：线性RNN梯度推导（BPTT）
模型定义（无偏置）：
$$h_t = W_{hh}h_{t-1} + W_{hx}x_t,\quad o_t = W_{oh}h_t$$
损失：$L=\frac12\sum_{t=1}^T (o_t-y_t)^2$
1. 单步输出梯度：$\frac{\partial L}{\partial o_t}=o_t-y_t$，$\frac{\partial L}{\partial h_t}=(o_t-y_t)W_{oh}^\top$
2. 时间反向传播递推：$\frac{\partial L}{\partial h_{t-1}} = W_{hh}^\top \frac{\partial L}{\partial h_t}$
3. 链式展开至所有权重梯度：
$$\frac{\partial h_t}{\partial W_{hh}} = \sum_{s=1}^t \left(W_{hh}\right)^{t-s} h_{s-1}$$
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L}{\partial h_t} \left( \sum_{s=1}^t W_{hh}^{t-s} h_{s-1} \right)$$

梯度消失/爆炸条件：
- 若 $W_{hh}$ 谱半径 $<1$，矩阵连乘指数衰减 → **梯度消失**
- 若 $W_{hh}$ 谱半径 $>1$，矩阵连乘指数放大 → **梯度爆炸**

### 3.2 编程题：手动实现RNN单元前向+反向（tanh激活）

In [2]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    # x_t: [B, D], h_prev: [B, H]
    pre_act = np.matmul(x_t, W_hx) + np.matmul(h_prev, W_hh) + b_h
    h_t = np.tanh(pre_act)
    cache = (x_t, h_prev, W_hx, W_hh, b_h, pre_act)
    return h_t, cache

def rnn_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, b_h, pre_act = cache
    # tanh导数: 1 - tanh²(z)
    dtanh = dh_next * (1 - np.tanh(pre_act) ** 2)
    db_h = np.sum(dtanh, axis=0, keepdims=True)
    dW_hx = np.matmul(x_t.T, dtanh)
    dW_hh = np.matmul(h_prev.T, dtanh)
    dx_t = np.matmul(dtanh, W_hx.T)
    dh_prev = np.matmul(dtanh, W_hh.T)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试
if __name__ == '__main__':
    B, D, H = 2, 3, 4
    x = np.random.randn(B, D)
    h_prev = np.random.randn(B, H)
    Whx = np.random.randn(D, H)
    Whh = np.random.randn(H, H)
    bh = np.zeros((1, H))
    h_t, cache = rnn_forward(x, h_prev, Whx, Whh, bh)
    dh_next = np.random.randn(B, H)
    dx, dh_prev, dWhx, dWhh, dbh = rnn_backward(dh_next, cache)
    print("h_t shape:", h_t.shape)
    print("dW_hh shape:", dWhh.shape)

h_t shape: (2, 4)
dW_hh shape: (4, 4)


## 4 高级循环神经网络
### 4.1 理论计算题：深度双向RNN总参数量
参数定义：$L$层数，输入维度$D$，隐层维度$H$，输出维度$O$
单层单向RNN参数：$W_{hx}(D,H)+W_{hh}(H,H)+b_h(H) = DH+H^2+H$
单层双向RNN（前向+后向）：$2(DH+H^2+H)$
第 $l\ge2$ 层双向RNN输入维度为 $2H$，单层参数：$2(2H\cdot H+H^2+H)=2(3H^2+H)$
总参数 = 第1层双向参数 + $(L-1)$层深层双向参数 + 输出层参数
$$
\begin{aligned}
\text{Total} &= 2(DH+H^2+H) + (L-1)\cdot 2(3H^2+H) + (2H\cdot O + O)
\end{aligned}
$$

### 4.2 编程题：双向RNN编码器（PyTorch）

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, bidirectional=True, batch_first=False)
    def forward(self, X):
        # X: [seq_len, batch, input_dim]
        out_seq, hn = self.rnn(X)
        # out_seq: [seq_len, batch, 2*hidden_dim] 每步拼接前向+后向
        # hn: [2, batch, hidden_dim]，拼接为全局序列表示
        h_global = torch.cat([hn[0], hn[1]], dim=-1)
        return out_seq, h_global

# 测试
if __name__ == '__main__':
    seq_len, batch, d_in, d_h = 5, 3, 4, 6
    x = torch.randn(seq_len, batch, d_in)
    encoder = BiRNNEncoder(d_in, d_h)
    seq_out, global_h = encoder(x)
    print("逐时间步拼接隐藏态 shape:", seq_out.shape)
    print("最终全局拼接隐藏态 shape:", global_h.shape)

逐时间步拼接隐藏态 shape: torch.Size([5, 3, 12])
最终全局拼接隐藏态 shape: torch.Size([3, 12])


## 5 嵌入向量
### 5.1 理论计算题：Skip-gram负采样损失
给定中心词向量 $v_c$，正上下文词向量 $u_o$，$K$个负样本 $u_{n_k}$，$\sigma(z)=\frac{1}{1+e^{-z}}$
单样本对数似然目标函数：
$$
\mathcal{L} = -\left[ \log\sigma(v_c^\top u_o) + \sum_{k=1}^K \log\sigma\left(-v_c^\top u_{n_k}\right) \right]
$$
负样本采样策略：按词频分布的 $3/4$ 次幂构建噪声分布，高频词采样概率更高。

### 5.2 编程题：CBOW完整Softmax前向与交叉熵损失

In [4]:
import numpy as np

def cbow_forward_loss(context_ids, target_id, W, W_out):
    """
    context_ids: list，单个样本上下文词索引
    target_id: 中心词索引
    W: [V, d] 输入嵌入矩阵
    W_out: [d, V] 输出权重矩阵
    返回：交叉熵损失标量
    """
    # 取上下文向量求平均
    ctx_vecs = W[context_ids]  # [context_size, d]
    h = np.mean(ctx_vecs, axis=0, keepdims=True)  # [1, d]
    # 计算得分+softmax
    score = np.matmul(h, W_out)  # [1, V]
    exp_score = np.exp(score - np.max(score, axis=-1, keepdims=True))
    prob = exp_score / np.sum(exp_score, axis=-1, keepdims=True)
    # 交叉熵
    log_p = -np.log(prob[0, target_id] + 1e-8)
    return log_p.item()

# 测试
if __name__ == '__main__':
    V, d, ctx_size = 10, 4, 3
    W = np.random.randn(V, d) * 0.01
    W_out = np.random.randn(d, V) * 0.01
    ctx = [1, 3, 5]
    target = 2
    loss = cbow_forward_loss(ctx, target, W, W_out)
    print("CBOW交叉熵损失:", loss)

CBOW交叉熵损失: 2.3024433004991183


## 6 注意力机制
### 6.1 理论计算题：缩放点积注意力
已知：$Q\in\mathbb{R}^{2\times4}, K\in\mathbb{R}^{3\times4}, V\in\mathbb{R}^{3\times5}, d_k=4, \sqrt{d_k}=2$
步骤1：计算原始得分矩阵 $S = QK^\top$，$S\in\mathbb{R}^{2\times3}$
步骤2：缩放得分 $\tilde{S} = S / \sqrt{d_k} = S/2$
步骤3：按行做Softmax得到注意力权重 $A = \text{softmax}(\tilde{S}),\ A\in\mathbb{R}^{2\times3}$
步骤4：加权求和输出 $\text{AttnOut} = A V \in\mathbb{R}^{2\times5}$

### 6.2 编程题：多头注意力 num_heads=2, d_model=4

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        # 投影矩阵 QKV
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
    def split_heads(self, x):
        # x: [seq_len, batch, d_model] -> [num_heads, seq_len, batch, d_k]
        seq_len, batch, _ = x.shape
        x = x.view(seq_len, batch, self.num_heads, self.d_k).permute(2,0,1,3)
        return x
    def forward(self, X):
        seq_len, batch, _ = X.shape
        # 线性投影
        Q = self.w_q(X)
        K = self.w_k(X)
        V = self.w_v(X)
        # 分头
        Qh = self.split_heads(Q)
        Kh = self.split_heads(K)
        Vh = self.split_heads(V)
        # 缩放点积注意力
        attn_score = torch.matmul(Qh, Kh.transpose(-1,-2)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn_weight = F.softmax(attn_score, dim=-1)
        attn_out_h = torch.matmul(attn_weight, Vh)
        # 拼接多头
        attn_out_h = attn_out_h.permute(1,2,0,3).contiguous()
        concat = attn_out_h.view(seq_len, batch, self.d_model)
        # 输出线性层
        out = self.w_o(concat)
        return out

# 测试
if __name__ == '__main__':
    seq_len, batch = 6, 2
    x = torch.randn(seq_len, batch, 4)
    mha = MultiHeadAttention(d_model=4, num_heads=2)
    res = mha(x)
    print("多头注意力输出shape:", res.shape)

多头注意力输出shape: torch.Size([6, 2, 4])
